# 03. Error Analysis & Stress Concentration Factor (SCF)

This notebook loads trained PI-DeepONet models from all 4 experimental arms and evaluates them against the **exact analytical Kirsch solution**.

We compute:
1. **L2 relative errors** for stress components (σ_xx, σ_yy, σ_xy)
2. **Stress Concentration Factor** accuracy (theoretical = 3.0)
3. **Stress field contour comparisons** (prediction vs analytical)

**Prerequisites**: Run `python kaggle_run_all.py` first to train all models.

In [ ]:
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Add paths
sys.path.append(os.path.abspath('../src'))
sys.path.append(os.path.abspath('../fem_baseline'))

from model import PIDeepONet
from physics import compute_pde_residuals
from analytical_kirsch import (
    analytical_kirsch_stress,
    stress_concentration_factor,
)

RESULTS_DIR = os.path.abspath('../results')

arms = ['baseline', 'rar_collocation', 'rar_load', 'rar_combined']
labels = {
    'baseline': 'Baseline (Uniform)',
    'rar_collocation': 'Collocation RAR',
    'rar_load': 'Load RAR',
    'rar_combined': 'Combined RAR',
}
colors = {
    'baseline': '#6C757D',
    'rar_collocation': '#0D6EFD',
    'rar_load': '#198754',
    'rar_combined': '#DC3545',
}

print(f'Theoretical SCF = {stress_concentration_factor():.4f}')

## 1. Load Trained Models

In [ ]:
models = {}
for arm in arms:
    # Try best model first, then final
    for suffix in ['best', 'final']:
        path = os.path.join(RESULTS_DIR, f'{arm}_{suffix}.pt')
        if os.path.exists(path):
            model = PIDeepONet(
                branch_layers=[100, 128, 128, 100],
                trunk_layers=[2, 128, 128, 50],
                num_outputs=2,
            )
            model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
            model.eval()
            models[arm] = model
            print(f'✓ Loaded {arm} ({suffix})')
            break
    else:
        print(f'⚠ No model found for {arm}')

if not models:
    print('\nNo trained models found. Run training first!')

## 2. Compute L2 Errors and SCF

In [ ]:
def relative_l2_error(pred, true):
    return np.linalg.norm(pred - true) / (np.linalg.norm(true) + 1e-10)

if models:
    R = 0.2
    T = 1.0
    L = 1.0
    true_scf = stress_concentration_factor(R, T)

    # Generate validation points
    np.random.seed(999)
    val_pts = []
    while len(val_pts) < 3000:
        pts = np.random.uniform(-L, L, (6000, 2))
        valid = pts[pts[:, 0]**2 + pts[:, 1]**2 >= R**2]
        val_pts.extend(valid.tolist())
    val_pts = np.array(val_pts[:3000])

    # Analytical solution
    sxx_true, syy_true, sxy_true = analytical_kirsch_stress(
        val_pts[:, 0], val_pts[:, 1], R=R, T=T
    )

    # Uniform load for evaluation
    load_uniform = torch.ones(1, 100, dtype=torch.float32) * T

    results = {}
    for arm, model in models.items():
        coords = torch.tensor(val_pts, dtype=torch.float32, requires_grad=True)
        pred = model(load_uniform, coords)
        u = pred[0, :, 0:1]
        v = pred[0, :, 1:2]
        _, _, stresses = compute_pde_residuals(coords, u, v)

        sxx_p = stresses['sigma_xx'].detach().numpy().squeeze()
        syy_p = stresses['sigma_yy'].detach().numpy().squeeze()
        sxy_p = stresses['sigma_xy'].detach().numpy().squeeze()

        # L2 errors
        l2_sxx = relative_l2_error(sxx_p, sxx_true)
        l2_syy = relative_l2_error(syy_p, syy_true)
        l2_sxy = relative_l2_error(sxy_p, sxy_true)

        # SCF at (0, R)
        scf_coord = torch.tensor([[0.0, R]], dtype=torch.float32, requires_grad=True)
        pred_scf = model(load_uniform, scf_coord)
        u_scf = pred_scf[0, :, 0:1]
        v_scf = pred_scf[0, :, 1:2]
        _, _, s_scf = compute_pde_residuals(scf_coord, u_scf, v_scf)
        predicted_scf = s_scf['sigma_xx'].item() / T

        results[arm] = {
            'L2(σ_xx)': l2_sxx, 'L2(σ_yy)': l2_syy, 'L2(σ_xy)': l2_sxy,
            'SCF': predicted_scf, 'SCF Error': abs(predicted_scf - true_scf),
        }

    # Display as table
    import pandas as pd
    df = pd.DataFrame(results).T
    df.index = [labels[a] for a in df.index]
    print(df.to_string(float_format='{:.4f}'.format))

## 3. SCF Bar Chart

In [ ]:
if models and results:
    fig, ax = plt.subplots(figsize=(10, 6))

    arm_names = list(results.keys())
    scf_vals = [results[a]['SCF'] for a in arm_names]
    bar_colors = [colors[a] for a in arm_names]
    bar_labels = [labels[a] for a in arm_names]

    bars = ax.bar(bar_labels, scf_vals, color=bar_colors, edgecolor='white', linewidth=2)
    ax.axhline(y=true_scf, color='black', linestyle='--', linewidth=2,
               label=f'Theoretical SCF = {true_scf:.1f}')

    # Add value labels on bars
    for bar, val in zip(bars, scf_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_ylabel('Stress Concentration Factor', fontsize=12)
    ax.set_title('SCF Prediction Accuracy by Experimental Arm', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_ylim(0, 4.0)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. L2 Error Comparison

In [ ]:
if models and results:
    fig, ax = plt.subplots(figsize=(10, 6))

    x = np.arange(len(arm_names))
    width = 0.25

    for i, stress_comp in enumerate(['L2(σ_xx)', 'L2(σ_yy)', 'L2(σ_xy)']):
        vals = [results[a][stress_comp] for a in arm_names]
        offset = (i - 1) * width
        bars = ax.bar(x + offset, vals, width, label=stress_comp, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(bar_labels, fontsize=10)
    ax.set_ylabel('Relative L2 Error', fontsize=12)
    ax.set_title('L2 Relative Errors by Stress Component', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_yscale('log')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()